# Inspect Aggregated Snow-Covered Area

HRU-level inspection of MOD10C1 snow-covered area. Mirrors `inspect_consolidated_snow_covered_area.ipynb`.

Source (see `catalog/variables.yml` → `snow_covered_area`):

- MOD10C1 v061: `Day_CMG_Snow_Cover` (SCA, native 0–100 integer scale), `Day_CMG_Clear_Index` (CI, native 0–100 integer scale).

The MOD10C1 aggregator follows the repo's transformation policy (`docs/architecture/transformation-pipeline.md`): pixel-defined operations — flag masks and the CI > 70 quality gate — run **pre-aggregation**; native variable names and units pass through. The `÷ 100` rescale to fractional `[0, 1]` is a linear conversion that happens **post-aggregation** in this notebook and in `targets/sca.py`.

See `docs/references/calibration-target-recipes.md` §5 for the SCA recipe context and the open CI-bounds methodology gap.

## Per-target conventions in this notebook

- The aggregated NC carries `Day_CMG_Snow_Cover` and `Day_CMG_Clear_Index` in their **native 0–100 integer scale**. The aggregator has already applied the per-pixel CI > 70 gate (TM 6-B10) and flag masks (107 lake ice, 111 night, 237 inland water, 239 ocean, 250 cloud-obscured water, 253 not mapped, 255 fill).
- This notebook applies `÷ 100` to display fractional `[0, 1]`.
- `valid_area_fraction` (in the aggregated NC) gives the HRU fraction whose pixels passed the CI gate. Use it as a coverage diagnostic alongside SCA.
- `Snow_Spatial_QA` is categorical 0–4 with flag codes — **NOT** a percentage CI. Earlier catalog versions confused it with `Day_CMG_Clear_Index`; the notebook never uses `Snow_Spatial_QA` quantitatively.
- Single-source target — no cross-source colour scale; fixed `vmin=0, vmax=1`.

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    daily_coverage_summary,
    discover_aggregated,
    find_best_day,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/aggregated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "snow_covered_area"
# Pick a winter day with both broad snow cover and clear-sky pixel coverage.
# Avoid early 2000 — the MOD10C1 v061 archive starts on 2000-02-24 because
# Terra MODIS systematic data begins late Feb 2000; any TARGET_DAY before
# then snaps to 2000-02-24 via .sel(method="nearest") and the inspection
# panels look near-empty. Use _helpers.find_best_day(ds, "Day_CMG_Snow_Cover",
# coverage_var="valid_area_fraction") to pick a different day automatically.
TARGET_DAY = "2000-12-22"
TARGET_YEAR = 2000
TARGET_MONTH = 12
TIME_SERIES_YEARS = range(2000, 2006)  # daily data — 6 years is enough
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}
SOURCES = {
    "mod10c1_v061": {
        "label": "MOD10C1 v061",
        "var": "Day_CMG_Snow_Cover",
        "ci_var": "Day_CMG_Clear_Index",
    },
}

MOD10C1_FLAGS = {107, 111, 237, 239, 250, 253, 255}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")


## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

In [ ]:
def _mask_flags(da: xr.DataArray) -> xr.DataArray:
    """Mask MOD10C1 flag values (>100) to NaN.

    The aggregator already applies this on its way through; this helper
    is for the validation cell, which reads the *raw* consolidated NC
    from the datastore and needs to mask flags itself.
    """
    return da.where(~da.isin(list(MOD10C1_FLAGS)))


def _to_fraction(da: xr.DataArray) -> xr.DataArray:
    """Native 0–100 integer scale → fractional [0, 1].

    The aggregator preserves the source's native scale per the repo's
    transformation policy (docs/architecture/transformation-pipeline.md);
    this rescale is the notebook/target-side responsibility.
    """
    return da / 100.0


def _monthly_mean_fraction(ds: xr.Dataset, var: str, year: int, month: int):
    """Monthly mean of an already-CI-filtered daily variable, in fractional [0, 1]."""
    da = ds[var].sel(time=slice(
        pd.Timestamp(year=year, month=month, day=1),
        pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    ))
    return _to_fraction(da).mean("time", skipna=True).to_pandas()


if not opened:
    print("No sources available; skipping native-unit maps.")
    monthly = None
else:
    ds, info = opened["mod10c1_v061"]
    da_day = _to_fraction(ds[info["var"]].sel(time=TARGET_DAY, method="nearest"))
    monthly = _monthly_mean_fraction(ds, info["var"], TARGET_YEAR, TARGET_MONTH)
    vaf = (
        ds["valid_area_fraction"].sel(time=TARGET_DAY, method="nearest").to_pandas()
        if "valid_area_fraction" in ds.data_vars
        else None
    )

    n_panels = 3 if vaf is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(8 * n_panels, 6), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, da_day.to_pandas(),
        vmin=0, vmax=1, cmap="Blues",
        title=f"Daily SCA (CI-filtered by aggregator)\n{TARGET_DAY}",
        units="fraction",
    )
    plot_hru_choropleth(
        axes[0, 1], fabric, monthly,
        vmin=0, vmax=1, cmap="Blues",
        title=f"Monthly mean SCA\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="fraction",
    )
    if vaf is not None:
        plot_hru_choropleth(
            axes[0, 2], fabric, vaf,
            vmin=0, vmax=1, cmap="viridis",
            title=f"valid_area_fraction (CI > 70 coverage)\n{TARGET_DAY}",
            units="fraction",
        )
    fig.suptitle(
        "Snow-covered area — aggregator pre-applies pixel-level CI > 70 + flag masks; ÷100 rescale here",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

### Reading the triptych

Three different time slices on the HRU fabric, all rescaled to fraction `[0, 1]`:

- **Left — Daily SCA on `TARGET_DAY`.** A single-day snapshot post-CI gate. NaN HRUs (light grey) are HRUs whose pixels all failed the CI > 70 filter that day — typically cloud-covered or low-quality returns. They are *not* no-snow HRUs. Visible snow is real, but the spatial extent on any one day is small.

- **Middle — Monthly mean SCA over `TARGET_YEAR-TARGET_MONTH`.** A NaN-skipping mean across the calendar month, so an HRU appears here as soon as it had at least one CI-passing observation in the month. This is the unit the calibration target ultimately consumes; it should look like climatology.

- **Right — `valid_area_fraction` on `TARGET_DAY`.** The fraction of each HRU's area where pixels passed the CI gate that day. It's a coverage diagnostic, not a snow value: bright = "we observed this HRU well today"; dark = "we did not".

The left and right panels share `TARGET_DAY` but the middle covers a whole month, so they will not visually align. **If the right panel is dark across most of CONUS, expect the left panel to look near-empty too — that's a coverage-poor day, not a no-snow day.** Use `find_best_day(ds, "Day_CMG_Snow_Cover", coverage_var="valid_area_fraction")` (optionally with `month=...`) to pick a day with both snow and good coverage.


## Monthly mean SCA

The aggregator already applied the CI > 70 gate per pixel before area-weighting to the HRU fabric, so the daily values in `Day_CMG_Snow_Cover` are NaN where pixel CI failed. The monthly mean is a NaN-skipping mean over the target month, rescaled to fractional `[0, 1]`. This is what the SCA target builder consumes.

In [ ]:
if opened and monthly is not None:
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, monthly,
        vmin=0, vmax=1, cmap="Blues",
        title=f"MOD10C1 v061 — monthly mean SCA (aggregator-CI-filtered)\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="fraction",
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()
elif opened:
    print("Monthly composite unavailable; see cell 7.")

### Why the monthly composite, not the daily snapshot, is the inspection unit

The TM 6-B10 SCA target reasons about daily fractional SCA at calibration time, but a single day is dominated by cloud cover — most HRUs will be NaN on any given day even with the CI > 70 gate. Averaging over the month with `skipna=True` lets each HRU "vote" with whatever CI-passing observations it had, smoothing the cloud-day mosaic.

Climatology cross-check (winter months — e.g. Dec/Jan/Feb):

- Northern Rockies, Cascades, northern Sierras, Adirondacks, northern New England: **near 1.0** (full or near-full snow cover all month).
- Northern Great Plains, upper Midwest, Lake-effect belt: **0.5–0.9** (variable cover, melt-and-redeposit cycles).
- Southern Appalachians, Mid-Atlantic, central Plains: **0.1–0.4** (a few snow events plus melt).
- Southern California, Arizona, Gulf Coast, Florida: **near 0.0** year-round (a non-zero monthly mean here is suspicious — bordering ocean pixels or flag mishandling).

A winter map that comes out predominantly near-zero is the canonical sign that the CI gate, flag mask, or `÷ 100` rescale broke upstream.


In [ ]:
if opened and monthly is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(monthly.dropna(), bins=60, histtype="step", linewidth=2, density=True)
    ax.set_xlabel("Snow-covered area (fraction)")
    ax.set_ylabel("Density")
    ax.set_title(f"HRU-level SCA distribution — {TARGET_YEAR}-{TARGET_MONTH:02d} (aggregator-CI-filtered)")
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

### Reading the SCA histogram

Expected shape for a winter month is **bimodal**:

- A peak near **0.0** — HRUs in low-snow regions (most of the Sun Belt, lower Great Plains, urban coastal HRUs).
- A peak near **1.0** — HRUs in persistent-cover regions (high-elevation mountain HRUs, far-northern CONUS).
- A "valley" between them — partial-coverage HRUs at mid-elevations, snow-line transitions, and edge-of-pack regions.

Useful red flags:

- **Unimodal near 0** in a winter month: the CI gate dropped most snowy pixels, or the rescale didn't run.
- **Unimodal near 1**: the gate is permissive (cloud-obscured pixels survived), or flag values bypassed the mask.
- **A spike at exactly 0 or 1** with nothing in between: integer-scale data leaked through without the `÷ 100` rescale.
- **A long plateau** between 0 and 1: typical for shoulder-season months (Mar/Nov), unusual mid-winter.


In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    ds_range = open_year_range(project_dir, "mod10c1_v061", TIME_SERIES_YEARS)
    try:
        id_dim = fabric_cfg["id_col"]
        sca_full = (
            ds_range["Day_CMG_Snow_Cover"]
            .sel({id_dim: list(rep_hrus.values())})
            .load()
        )
    finally:
        ds_range.close()

    # Aggregator already applied the CI > 70 pixel gate; just rescale to fractional [0, 1].
    sca_full = sca_full / 100.0

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        ax.plot(sca_full.time, sca_full.sel({id_dim: hru_id}).values, linewidth=0.8)
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("SCA fraction")
        ax.set_ylim(0, 1)
    fig.suptitle(
        f"SCA at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)} "
        f"(aggregator-CI-filtered, fractional)"
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

### Representative-HRU seasonality check

Each panel is a multi-year daily SCA time series at one HRU, picked to span very different snow regimes. The expected signatures:

- **Olympic Peninsula (PNW mountains)** — Deep, fast-onset winter peaks (Nov–Dec), persistent through spring at higher elevations, summers near 0. Strong inter-year variability driven by Pacific storm tracks.
- **Iowa cropland (Midwest)** — Sharper winter peaks (Dec–Feb), sometimes near 1.0, with abrupt early-spring melt; near-zero May–Oct. Years with strong polar vortex events show extended persistence.
- **Phoenix metro (arid SW)** — Should be near-zero year-round. Any non-zero signal is a rare desert dusting (real but transient) or, more often, an HRU whose footprint includes a high-elevation edge — sanity-check by inspecting the HRU geometry.
- **Southern Appalachians (Eastern forest)** — Short pulses of snow lasting days to a week or two, smaller amplitude than mountain HRUs because forest canopy and elevation both temper accumulation.

Visual gaps in the trace are **NaN days from the CI > 70 gate**, not missing observations — the satellite passed but cloud cover prevented confident retrieval. Long stretches of NaN in winter (especially PNW) are normal; in summer they're rarer because clear-sky frequency is higher.


In [ ]:
if opened and monthly is not None:
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    n_nan = nan_hru_count(monthly)
    print(f"MOD10C1 v061: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
    plot_nan_hrus(
        axes[0, 0],
        fabric,
        monthly,
        title=f"NaN HRUs (red) — {n_nan} of {len(fabric)} (aggregator-CI-filtered monthly)",
    )
    fig.suptitle("Coverage gaps — NN-filled in normalize/", fontsize=12, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

### Coverage-gap interpretation

Red HRUs are NaN in the **monthly composite** — that means they had zero days during the entire month with any CI-passing pixel coverage. Two distinct populations land here:

- **Persistent overcast HRUs.** Cloud regimes are spatially correlated, so coverage gaps cluster geographically (think a multi-week atmospheric river over the PNW, or a stalled stratus deck off the southeast Atlantic coast). These are *no-data* HRUs, not *no-snow* HRUs.
- **Geometry-narrow HRUs.** Long, thin, or coast-clipping HRU polygons whose footprint rarely intersects a clear-sky 0.05° pixel center. These can be persistently NaN even in clear-weather regions.

Downstream handling is in `normalize/` — NaN HRUs are filled by **nearest-neighbor lookup** from a CI-passing neighbor before the SCA target builder runs. NN-fill is appropriate where coverage gaps are spatially correlated (the neighbor is climatologically similar); it's a poorer choice when the gap is large enough to span a regime boundary, but those cases are rare in monthly composites.

If the NaN fraction is climbing significantly above ~5% of fabric in winter months, suspect either an aggregator regression or a fabric/source-grid registration drift. Cross-check `valid_area_fraction` distribution before chasing aggregator code.


In [ ]:
def _gridded_mean_sca(year, month):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox; HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal.
    """
    path = datastore_dir / "mod10c1_v061" / f"mod10c1_v061_{year}.nc"
    if not path.exists():
        return None, f"missing consolidated NC at {path}"
    with xr.open_dataset(path) as ds:
        if "Day_CMG_Clear_Index" not in ds.data_vars:
            return None, "Day_CMG_Clear_Index missing in consolidated NC"
        sca = _mask_flags(ds["Day_CMG_Snow_Cover"]) / 100.0
        ci = _mask_flags(ds["Day_CMG_Clear_Index"]) / 100.0
        sca_valid = sca.where(ci > 0.70).sel(time=slice(
            pd.Timestamp(year=year, month=month, day=1),
            pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
        )).load()
    return float(sca_valid.mean(skipna=True).item()), None


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
if opened and monthly is not None:
    agg_mean = area_weighted_mean(monthly, fabric)
    gridded_mean, reason = _gridded_mean_sca(TARGET_YEAR, TARGET_MONTH)
    if gridded_mean is None:
        print(f"{'MOD10C1 v061':<35} {agg_mean:>12.4f}  {'SKIP':>12} ({reason})")
    else:
        diff = agg_mean - gridded_mean
        pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
        print(f"{'MOD10C1 v061':<35} {agg_mean:>12.4f} {gridded_mean:>12.4f} {diff:>12.4f} {pct:>7.2f}%")
else:
    print("Validation skipped — CI-filtered composite unavailable.")

### Reading the aggregated-vs-gridded validation row

The two means are **not directly comparable** by construction:

- **Aggregated** is the Albers-area-weighted mean over fabric HRUs only (CONUS plus contributing HUC-2 buffer).
- **Gridded** is the unweighted mean of the consolidated NC bbox, which extends past the fabric into ocean, Canada, Mexico, and Great Lakes pixels. After flag-masking and CI gating, the gridded mean is dominated by whatever pixels survived in the bbox-but-not-fabric edges.

Expected outcomes:

- **% diff between -30% and +30%** is normal. The aggregated value usually runs *higher* in winter (the fabric is biased toward higher latitudes than the bbox average, which dilutes with low-latitude or oceanic edges).
- **|% diff| > 50%** suggests a real bug — usually a missed `÷ 100` rescale somewhere, a CI-gate inversion, or flag values surviving into the mean. Check the catalog units, then `valid_area_fraction`, before suspecting the aggregator's weights.
- **`SKIP (missing consolidated NC at …)`** simply means the datastore doesn't have the raw daily MOD10C1 file for `TARGET_YEAR`. The aggregated value still validates against itself via the upstream cells; the gridded cross-check is best-effort.

This row is a coarse *sanity check*, not a unit test — pass/fail here doesn't gate the calibration target.


## Why this is a coverage / quality diagnostic, not a cross-source comparison

MOD10C1 is the only SCA source for the calibration target, so this notebook isn't comparing across sources — it's checking that aggregated values look sane on the HRU fabric.

- **The CI gate is critical.** TM 6-B10 keeps pixels where confidence index > 70. With it, a winter monthly mean is the area-weighted mean of high-confidence pixels' SCAs. Without it, the mean would be dominated by cloud-obscured-water and night flag codes that survive too-narrow flag masks.
- **The gate is applied per pixel, not per HRU.** This matters: an HRU with 50% high-confidence snowy pixels and 50% low-confidence cloud pixels gives a per-pixel-gated answer (mean = snowy half) that differs from a per-HRU-gated answer (contaminated mean, or whole HRU dropped). See `docs/architecture/transformation-pipeline.md` for the commutativity argument.
- **`valid_area_fraction`** is the HRU-fraction with valid CI-passing pixels; use it as a coverage companion to SCA.
- **`Snow_Spatial_QA` is categorical, not a percent CI.** Earlier catalog versions confused it with `Day_CMG_Clear_Index`. The notebook never uses it quantitatively.

**Calibration target implication.** TM 6-B10 derives upper/lower bounds from the daily SCA value and the associated CI. The exact formula is unconfirmed (PRMSobjfun.f not publicly available — open methodology gap). What this notebook checks is that the input to that formula — daily fractional SCA on the HRU fabric, already CI-filtered by the aggregator — is sane.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()